In [ ]:
import pandas as pd
import json
import ast

# Read the CSV files
OL_df = pd.read_csv('../../../../data/augmentation/decrease_naturalness/experiments/original_nl_new_sql_results-with_penalty.csv')
LO_df = pd.read_csv('../../../../data/augmentation/decrease_naturalness/experiments/new_nl_original_sql_results-with_penalty.csv')
LL_df = pd.read_csv('../../../../data/augmentation/decrease_naturalness/experiments/new_nl_new_sql_results-with_penalty.csv')

print(f"OL_df shape: {OL_df.shape}")
print(f"LO_df shape: {LO_df.shape}")
print(f"LL_df shape: {LL_df.shape}")



In [ ]:
# Function to extract EX value from the EXECUTION_ACCURACY column
def extract_ex(value):
    try:
        if pd.isna(value):
            return None
        # Parse the string as a dictionary
        if isinstance(value, str):
            data = ast.literal_eval(value)
        else:
            data = value
        return data.get('EX', None)
    except:
        return None

# Extract EX values for each dataframe
OL_df['EX'] = OL_df['EXECUTION_ACCURACY'].apply(extract_ex)
LO_df['EX'] = LO_df['EXECUTION_ACCURACY'].apply(extract_ex)
LL_df['EX'] = LL_df['EXECUTION_ACCURACY'].apply(extract_ex)

print("\nEX value distribution:")
print(f"OL_df EX values: {OL_df['EX'].value_counts()}")
print(f"LO_df EX values: {LO_df['EX'].value_counts()}")
print(f"LL_df EX values: {LL_df['EX'].value_counts()}")


In [ ]:
# Create a key to match rows across dataframes (using system, db_name, question_id)
# Key includes system to ensure we only match within the same system
OL_df['key'] = OL_df['system'] + '|' + OL_df['db_name'] + '|' + OL_df['question_id'].astype(str)
LO_df['key'] = LO_df['system'] + '|' + LO_df['db_name'] + '|' + LO_df['question_id'].astype(str)
LL_df['key'] = LL_df['system'] + '|' + LL_df['db_name'] + '|' + LL_df['question_id'].astype(str)

# Get unique systems
systems = sorted(LL_df['system'].unique().tolist())
print("="*80)
print("PER-SYSTEM ANALYSIS")
print("="*80)
print(f"\nAnalyzing three scenarios per system:")
print(f"  1. LL EX=1 AND OL EX=0 (benefits from naturalness decrease in NL)")
print(f"  2. LL EX=1 AND LO EX=0 (benefits from naturalness decrease in SQL)")
print(f"  3. LL EX=1 AND OL EX=0 AND LO EX=0 (requires BOTH to succeed)")
print("="*80)



In [ ]:
# SCENARIO 1: LL EX=1 AND OL EX=0 (PER SYSTEM)
print("\n" + "="*80)
print("SCENARIO 1: LL EX=1 AND OL EX=0")
print("="*80)
print("Description: Queries that succeed with new NL+new SQL but fail with original NL+new SQL")
print("Interpretation: These queries benefit from naturalness decrease in NL\n")

scenario_1_all_qids = []

for system in systems:
    print(f"\n{'='*80}")
    print(f"SYSTEM: {system.upper()}")
    print(f"{'='*80}")

    # Filter by system
    sys_LL = LL_df[LL_df['system'] == system]
    sys_OL = OL_df[OL_df['system'] == system]

    sys_LL_ex_1_keys = set(sys_LL[sys_LL['EX'] == 1]['key'].tolist())
    sys_OL_ex_0_keys = set(sys_OL[sys_OL['EX'] == 0]['key'].tolist())

    sys_LL_ex_1_count = len(sys_LL_ex_1_keys)
    sys_OL_ex_0_count = len(sys_OL_ex_0_keys)

    scenario_1_sys_keys = sys_LL_ex_1_keys & sys_OL_ex_0_keys

    if len(scenario_1_sys_keys) > 0:
        scenario_1_sys_df = sys_LL[sys_LL['key'].isin(scenario_1_sys_keys)][['system', 'db_name', 'question_id', 'question']].copy()
        scenario_1_sys_qids = sorted(scenario_1_sys_df['question_id'].unique().tolist())
        scenario_1_all_qids.extend([(system, qid) for qid in scenario_1_sys_qids])

        print(f"\nMatching queries: {len(scenario_1_sys_keys)}")
        print(f"  • Out of {sys_LL_ex_1_count} LL successes → {len(scenario_1_sys_keys)} failed in OL ({len(scenario_1_sys_keys)/sys_LL_ex_1_count*100:.1f}%)")
        print(f"  • Out of {sys_OL_ex_0_count} OL failures → {len(scenario_1_sys_keys)} succeeded in LL ({len(scenario_1_sys_keys)/sys_OL_ex_0_count*100:.1f}%)")

        print(f"\nQuestion IDs ({len(scenario_1_sys_qids)}):")
        print(scenario_1_sys_qids)

        print(f"\nTop databases:")
        db_counts = scenario_1_sys_df.groupby('db_name').size().sort_values(ascending=False).head(5)
        for db, count in db_counts.items():
            print(f"  • {db}: {count}")
    else:
        print(f"\nNo queries found for {system}")


In [ ]:
# SCENARIO 2: LL EX=1 AND LO EX=0 (PER SYSTEM)
print("\n" + "="*80)
print("SCENARIO 2: LL EX=1 AND LO EX=0")
print("="*80)
print("Description: Queries that succeed with new NL+new SQL but fail with new NL+original SQL")
print("Interpretation: These queries benefit from naturalness decrease in SQL\n")

scenario_2_all_qids = []

for system in systems:
    print(f"\n{'='*80}")
    print(f"SYSTEM: {system.upper()}")
    print(f"{'='*80}")

    # Filter by system
    sys_LL = LL_df[LL_df['system'] == system]
    sys_LO = LO_df[LO_df['system'] == system]

    sys_LL_ex_1_keys = set(sys_LL[sys_LL['EX'] == 1]['key'].tolist())
    sys_LO_ex_0_keys = set(sys_LO[sys_LO['EX'] == 0]['key'].tolist())

    sys_LL_ex_1_count = len(sys_LL_ex_1_keys)
    sys_LO_ex_0_count = len(sys_LO_ex_0_keys)

    scenario_2_sys_keys = sys_LL_ex_1_keys & sys_LO_ex_0_keys

    if len(scenario_2_sys_keys) > 0:
        scenario_2_sys_df = sys_LL[sys_LL['key'].isin(scenario_2_sys_keys)][['system', 'db_name', 'question_id', 'question']].copy()
        scenario_2_sys_qids = sorted(scenario_2_sys_df['question_id'].unique().tolist())
        scenario_2_all_qids.extend([(system, qid) for qid in scenario_2_sys_qids])

        print(f"\nMatching queries: {len(scenario_2_sys_keys)}")
        print(f"  • Out of {sys_LL_ex_1_count} LL successes → {len(scenario_2_sys_keys)} failed in LO ({len(scenario_2_sys_keys)/sys_LL_ex_1_count*100:.1f}%)")
        print(f"  • Out of {sys_LO_ex_0_count} LO failures → {len(scenario_2_sys_keys)} succeeded in LL ({len(scenario_2_sys_keys)/sys_LO_ex_0_count*100:.1f}%)")

        print(f"\nQuestion IDs ({len(scenario_2_sys_qids)}):")
        print(scenario_2_sys_qids)

        print(f"\nTop databases:")
        db_counts = scenario_2_sys_df.groupby('db_name').size().sort_values(ascending=False).head(5)
        for db, count in db_counts.items():
            print(f"  • {db}: {count}")
    else:
        print(f"\nNo queries found for {system}")


In [ ]:
# SCENARIO 3: LL EX=1 AND OL EX=0 AND LO EX=0 (PER SYSTEM)
print("\n" + "="*80)
print("SCENARIO 3: LL EX=1 AND OL EX=0 AND LO EX=0")
print("="*80)
print("Description: Queries that succeed ONLY with new NL+new SQL, fail with both other combinations")
print("Interpretation: These queries require BOTH NL and SQL naturalness decrease to succeed\n")

scenario_3_all_qids = []

for system in systems:
    print(f"\n{'='*80}")
    print(f"SYSTEM: {system.upper()}")
    print(f"{'='*80}")

    # Filter by system
    sys_LL = LL_df[LL_df['system'] == system]
    sys_OL = OL_df[OL_df['system'] == system]
    sys_LO = LO_df[LO_df['system'] == system]

    sys_LL_ex_1_keys = set(sys_LL[sys_LL['EX'] == 1]['key'].tolist())
    sys_OL_ex_0_keys = set(sys_OL[sys_OL['EX'] == 0]['key'].tolist())
    sys_LO_ex_0_keys = set(sys_LO[sys_LO['EX'] == 0]['key'].tolist())

    sys_LL_ex_1_count = len(sys_LL_ex_1_keys)
    sys_OL_ex_0_count = len(sys_OL_ex_0_keys)
    sys_LO_ex_0_count = len(sys_LO_ex_0_keys)

    scenario_3_sys_keys = sys_LL_ex_1_keys & sys_OL_ex_0_keys & sys_LO_ex_0_keys

    if len(scenario_3_sys_keys) > 0:
        scenario_3_sys_df = sys_LL[sys_LL['key'].isin(scenario_3_sys_keys)][['system', 'db_name', 'question_id', 'question']].copy()
        scenario_3_sys_qids = sorted(scenario_3_sys_df['question_id'].unique().tolist())
        scenario_3_all_qids.extend([(system, qid) for qid in scenario_3_sys_qids])

        print(f"\nMatching queries: {len(scenario_3_sys_keys)}")
        print(f"  • Out of {sys_LL_ex_1_count} LL successes → {len(scenario_3_sys_keys)} failed in BOTH OL & LO ({len(scenario_3_sys_keys)/sys_LL_ex_1_count*100:.1f}%)")
        print(f"  • Out of {sys_OL_ex_0_count} OL failures → {len(scenario_3_sys_keys)} succeeded in LL ({len(scenario_3_sys_keys)/sys_OL_ex_0_count*100:.1f}%)")
        print(f"  • Out of {sys_LO_ex_0_count} LO failures → {len(scenario_3_sys_keys)} succeeded in LL ({len(scenario_3_sys_keys)/sys_LO_ex_0_count*100:.1f}%)")

        print(f"\nQuestion IDs ({len(scenario_3_sys_qids)}):")
        print(scenario_3_sys_qids)

        print(f"\nTop databases:")
        db_counts = scenario_3_sys_df.groupby('db_name').size().sort_values(ascending=False).head(5)
        for db, count in db_counts.items():
            print(f"  • {db}: {count}")
    else:
        print(f"\nNo queries found for {system}")


In [ ]:
# SUMMARY TABLE - Numbers per Scenario per System
print("\n" + "="*80)
print("SUMMARY: SCENARIO COUNTS PER SYSTEM")
print("="*80)

summary_rows = []

for system in systems:
    # Filter by system
    sys_LL = LL_df[LL_df['system'] == system]
    sys_OL = OL_df[OL_df['system'] == system]
    sys_LO = LO_df[LO_df['system'] == system]

    sys_LL_ex_1_keys = set(sys_LL[sys_LL['EX'] == 1]['key'].tolist())
    sys_OL_ex_0_keys = set(sys_OL[sys_OL['EX'] == 0]['key'].tolist())
    sys_LO_ex_0_keys = set(sys_LO[sys_LO['EX'] == 0]['key'].tolist())

    # Calculate scenarios
    scenario_1_count = len(sys_LL_ex_1_keys & sys_OL_ex_0_keys)
    scenario_2_count = len(sys_LL_ex_1_keys & sys_LO_ex_0_keys)
    scenario_3_count = len(sys_LL_ex_1_keys & sys_OL_ex_0_keys & sys_LO_ex_0_keys)

    summary_rows.append({
        'System': system.upper(),
        'Scenario_1 (LL=1 & OL=0)': scenario_1_count,
        'Scenario_2 (LL=1 & LO=0)': scenario_2_count,
        'Scenario_3 (LL=1 & OL=0 & LO=0)': scenario_3_count,
        'Total_LL_Successes': len(sys_LL_ex_1_keys),
        'Total_OL_Failures': len(sys_OL_ex_0_keys),
        'Total_LO_Failures': len(sys_LO_ex_0_keys)
    })

summary_table = pd.DataFrame(summary_rows)
print("\n")
print(summary_table.to_string(index=False))

# Add totals
print("\n" + "-"*80)
print("TOTALS ACROSS ALL SYSTEMS:")
print(f"  Scenario 1: {summary_table['Scenario_1 (LL=1 & OL=0)'].sum()}")
print(f"  Scenario 2: {summary_table['Scenario_2 (LL=1 & LO=0)'].sum()}")
print(f"  Scenario 3: {summary_table['Scenario_3 (LL=1 & OL=0 & LO=0)'].sum()}")

# Find common questions across systems per scenario
print("\n" + "-"*80)
print("COMMON QUESTIONS FAILED ACROSS SYSTEMS:")
print("-"*80)

# Scenario 1: Find question_ids that appear in multiple systems
scenario_1_qids_by_system = {}
for system, qid in scenario_1_all_qids:
    if qid not in scenario_1_qids_by_system:
        scenario_1_qids_by_system[qid] = []
    scenario_1_qids_by_system[qid].append(system)

common_scenario_1 = {qid: systems_list for qid, systems_list in scenario_1_qids_by_system.items() if len(systems_list) > 1}
print(f"\nScenario 1 (LL=1 & OL=0) - Questions failed in multiple systems:")
if common_scenario_1:
    print(f"  Total common questions: {len(common_scenario_1)}")
    for qid, systems_list in sorted(common_scenario_1.items()):
        print(f"    • Question {qid}: {', '.join(sorted(systems_list))} ({len(systems_list)} systems)")
else:
    print(f"  No questions failed across multiple systems")

# Scenario 2: Find question_ids that appear in multiple systems
scenario_2_qids_by_system = {}
for system, qid in scenario_2_all_qids:
    if qid not in scenario_2_qids_by_system:
        scenario_2_qids_by_system[qid] = []
    scenario_2_qids_by_system[qid].append(system)

common_scenario_2 = {qid: systems_list for qid, systems_list in scenario_2_qids_by_system.items() if len(systems_list) > 1}
print(f"\nScenario 2 (LL=1 & LO=0) - Questions failed in multiple systems:")
if common_scenario_2:
    print(f"  Total common questions: {len(common_scenario_2)}")
    for qid, systems_list in sorted(common_scenario_2.items()):
        print(f"    • Question {qid}: {', '.join(sorted(systems_list))} ({len(systems_list)} systems)")
else:
    print(f"  No questions failed across multiple systems")

# Scenario 3: Find question_ids that appear in multiple systems
scenario_3_qids_by_system = {}
for system, qid in scenario_3_all_qids:
    if qid not in scenario_3_qids_by_system:
        scenario_3_qids_by_system[qid] = []
    scenario_3_qids_by_system[qid].append(system)

common_scenario_3 = {qid: systems_list for qid, systems_list in scenario_3_qids_by_system.items() if len(systems_list) > 1}
print(f"\nScenario 3 (LL=1 & OL=0 & LO=0) - Questions failed in multiple systems:")
if common_scenario_3:
    print(f"  Total common questions: {len(common_scenario_3)}")
    for qid, systems_list in sorted(common_scenario_3.items()):
        print(f"    • Question {qid}: {', '.join(sorted(systems_list))} ({len(systems_list)} systems)")
else:
    print(f"  No questions failed across multiple systems")

print("="*80)


In [ ]:
# SCENARIO 1: Question IDs per System
print("="*80)
print("SCENARIO 1: QUESTION IDs PER SYSTEM (LL=1 & OL=0)")
print("="*80)

scenario_1_by_system = {}
for system in systems:
    # Filter by system
    sys_LL = LL_df[LL_df['system'] == system]
    sys_OL = OL_df[OL_df['system'] == system]

    sys_LL_ex_1_keys = set(sys_LL[sys_LL['EX'] == 1]['key'].tolist())
    sys_OL_ex_0_keys = set(sys_OL[sys_OL['EX'] == 0]['key'].tolist())

    scenario_1_sys_keys = sys_LL_ex_1_keys & sys_OL_ex_0_keys

    if len(scenario_1_sys_keys) > 0:
        scenario_1_sys_df = sys_LL[sys_LL['key'].isin(scenario_1_sys_keys)][['system', 'db_name', 'question_id']].copy()
        scenario_1_sys_qids = sorted(scenario_1_sys_df['question_id'].unique().tolist())
        scenario_1_by_system[system] = scenario_1_sys_qids

        print(f"\n{system.upper()}: {len(scenario_1_sys_qids)} questions")
        print(scenario_1_sys_qids)
    else:
        scenario_1_by_system[system] = []
        print(f"\n{system.upper()}: No questions")

print("="*80)


In [ ]:
# SCENARIO 2: Question IDs per System
print("="*80)
print("SCENARIO 2: QUESTION IDs PER SYSTEM (LL=1 & LO=0)")
print("="*80)

scenario_2_by_system = {}
for system in systems:
    # Filter by system
    sys_LL = LL_df[LL_df['system'] == system]
    sys_LO = LO_df[LO_df['system'] == system]

    sys_LL_ex_1_keys = set(sys_LL[sys_LL['EX'] == 1]['key'].tolist())
    sys_LO_ex_0_keys = set(sys_LO[sys_LO['EX'] == 0]['key'].tolist())

    scenario_2_sys_keys = sys_LL_ex_1_keys & sys_LO_ex_0_keys

    if len(scenario_2_sys_keys) > 0:
        scenario_2_sys_df = sys_LL[sys_LL['key'].isin(scenario_2_sys_keys)][['system', 'db_name', 'question_id']].copy()
        scenario_2_sys_qids = sorted(scenario_2_sys_df['question_id'].unique().tolist())
        scenario_2_by_system[system] = scenario_2_sys_qids

        print(f"\n{system.upper()}: {len(scenario_2_sys_qids)} questions")
        print(scenario_2_sys_qids)
    else:
        scenario_2_by_system[system] = []
        print(f"\n{system.upper()}: No questions")

print("="*80)


In [ ]:
# SCENARIO 3: Question IDs per System
print("="*80)
print("SCENARIO 3: QUESTION IDs PER SYSTEM (LL=1 & OL=0 & LO=0)")
print("="*80)

scenario_3_by_system = {}
for system in systems:
    # Filter by system
    sys_LL = LL_df[LL_df['system'] == system]
    sys_OL = OL_df[OL_df['system'] == system]
    sys_LO = LO_df[LO_df['system'] == system]

    sys_LL_ex_1_keys = set(sys_LL[sys_LL['EX'] == 1]['key'].tolist())
    sys_OL_ex_0_keys = set(sys_OL[sys_OL['EX'] == 0]['key'].tolist())
    sys_LO_ex_0_keys = set(sys_LO[sys_LO['EX'] == 0]['key'].tolist())

    scenario_3_sys_keys = sys_LL_ex_1_keys & sys_OL_ex_0_keys & sys_LO_ex_0_keys

    if len(scenario_3_sys_keys) > 0:
        scenario_3_sys_df = sys_LL[sys_LL['key'].isin(scenario_3_sys_keys)][['system', 'db_name', 'question_id']].copy()
        scenario_3_sys_qids = sorted(scenario_3_sys_df['question_id'].unique().tolist())
        scenario_3_by_system[system] = scenario_3_sys_qids

        print(f"\n{system.upper()}: {len(scenario_3_sys_qids)} questions")
        print(scenario_3_sys_qids)
    else:
        scenario_3_by_system[system] = []
        print(f"\n{system.upper()}: No questions")

print("="*80)


In [ ]:
# AGGREGATE ALL QUESTION IDs FROM ALL SCENARIOS AND SYSTEMS
print("="*80)
print("ALL QUESTION IDs ACROSS ALL SCENARIOS AND SYSTEMS")
print("="*80)

# Collect all unique question IDs from all scenarios
all_question_ids = set()

# Add from scenario 1
for system, qid in scenario_1_all_qids:
    all_question_ids.add(qid)

# Add from scenario 2
for system, qid in scenario_2_all_qids:
    all_question_ids.add(qid)

# Add from scenario 3
for system, qid in scenario_3_all_qids:
    all_question_ids.add(qid)

# Convert to sorted list
all_question_ids_list = sorted(list(all_question_ids))

print(f"\nTotal unique question IDs across all scenarios and systems: {len(all_question_ids_list)}")
print(f"\nBreakdown:")
print(f"  - Scenario 1 total appearances: {len(scenario_1_all_qids)} (system-question pairs)")
print(f"  - Scenario 2 total appearances: {len(scenario_2_all_qids)} (system-question pairs)")
print(f"  - Scenario 3 total appearances: {len(scenario_3_all_qids)} (system-question pairs)")
print(f"  - Total appearances: {len(scenario_1_all_qids) + len(scenario_2_all_qids) + len(scenario_3_all_qids)} (system-question pairs)")

print(f"\n{'='*80}")
print("COMPLETE LIST OF ALL UNIQUE QUESTION IDs:")
print(f"{'='*80}")
print(all_question_ids_list)

# Also create a detailed breakdown showing which scenarios each question appears in
print(f"\n{'='*80}")
print("QUESTION ID BREAKDOWN BY SCENARIO MEMBERSHIP:")
print(f"{'='*80}")

# Create dictionaries to track which scenarios each question ID appears in
qid_scenarios = {}

for system, qid in scenario_1_all_qids:
    if qid not in qid_scenarios:
        qid_scenarios[qid] = {'systems': set(), 'scenarios': set()}
    qid_scenarios[qid]['systems'].add(system)
    qid_scenarios[qid]['scenarios'].add(1)

for system, qid in scenario_2_all_qids:
    if qid not in qid_scenarios:
        qid_scenarios[qid] = {'systems': set(), 'scenarios': set()}
    qid_scenarios[qid]['systems'].add(system)
    qid_scenarios[qid]['scenarios'].add(2)

for system, qid in scenario_3_all_qids:
    if qid not in qid_scenarios:
        qid_scenarios[qid] = {'systems': set(), 'scenarios': set()}
    qid_scenarios[qid]['systems'].add(system)
    qid_scenarios[qid]['scenarios'].add(3)

# Print breakdown
print(f"\nQuestions appearing in multiple scenarios:")
multi_scenario_count = 0
for qid in sorted(qid_scenarios.keys()):
    scenarios = sorted(list(qid_scenarios[qid]['scenarios']))
    systems = sorted(list(qid_scenarios[qid]['systems']))
    if len(scenarios) > 1:
        multi_scenario_count += 1
        print(f"  • Question {qid}: Scenarios {scenarios}, Systems {systems}")

print(f"\nTotal questions in multiple scenarios: {multi_scenario_count}")
print(f"Total questions in only one scenario: {len(qid_scenarios) - multi_scenario_count}")

print("="*80)


---------